<a href="https://colab.research.google.com/github/MudasirH-coder/DeveloperHub-Corporation-1/blob/main/Insurance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)


# 1. LOAD DATA


df = pd.read_csv("/content/insurance.csv")

print("="*50)
print("DATASET SHAPE")
print("="*50)
print(df.shape)

print("\n")


# 2. EDA


print("="*50)
print("FIRST 5 ROWS")
print("="*50)
print(df.head())

print("\n")

print("="*50)
print("DATA TYPES")
print("="*50)
print(df.dtypes)

print("\n")

print("="*50)
print("MISSING VALUES")
print("="*50)
print(df.isnull().sum())

print("\n")

print("="*50)
print("DUPLICATES")
print("="*50)
print(df.duplicated().sum())

print("\n")

print("="*50)
print("STATISTICAL SUMMARY")
print("="*50)
print(df.describe())


# 3. DATA CLEANING


# Remove duplicates if any

df = df.drop_duplicates()

print("\nShape After Removing Duplicates:")
print(df.shape)


# 4. FEATURE ENGINEERING


# BMI Category

df["bmi_category"] = pd.cut(
    df["bmi"],
    bins=[0, 18.5, 25, 30, 100],
    labels=["Underweight", "Normal", "Overweight", "Obese"]
)

# Smoker + BMI Interaction

df["bmi_smoker_interaction"] = (
    df["bmi"] *
    (df["smoker"] == "yes").astype(int)
)


# 5. ENCODING CATEGORICAL FEATURES


df = pd.get_dummies(
    df,
    columns=["sex", "smoker", "region", "bmi_category"],
    drop_first=True
)


# 6. DEFINE FEATURES & TARGET


X = df.drop("charges", axis=1)
y = df["charges"]

print("\n")
print("FEATURES:")
print(X.columns.tolist())


# 7. TRAIN TEST SPLIT


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)


# 8. FEATURE SCALING


scaler = StandardScaler()

numerical_cols = [
    "age",
    "bmi",
    "children",
    "bmi_smoker_interaction"
]

X_train[numerical_cols] = scaler.fit_transform(
    X_train[numerical_cols]
)

X_test[numerical_cols] = scaler.transform(
    X_test[numerical_cols]
)


# 9. TRAIN LINEAR REGRESSION


model = LinearRegression()

model.fit(X_train, y_train)


# 10. PREDICTIONS


y_pred = model.predict(X_test)


# 11. EVALUATION


r2 = r2_score(y_test, y_pred)

mae = mean_absolute_error(y_test, y_pred)

mse = mean_squared_error(y_test, y_pred)

rmse = np.sqrt(mse)

print("\n")
print("="*50)
print("MODEL PERFORMANCE")
print("="*50)

print(f"R² Score  : {r2:.4f}")
print(f"MAE       : {mae:.2f}")
print(f"MSE       : {mse:.2f}")
print(f"RMSE      : {rmse:.2f}")


# 12. COEFFICIENTS


coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_
})

coefficients = coefficients.sort_values(
    by="Coefficient",
    ascending=False
)

print("\n")
print("="*50)
print("FEATURE IMPORTANCE")
print("="*50)


print(coefficients)


# 13. INTERCEPT


print("\nIntercept:")
print(model.intercept_)


# 14. SAMPLE PREDICTION


sample_prediction = model.predict(
    X_test.iloc[[0]]
)

print("\nSample Prediction:")
print(sample_prediction)

print("\nActual Value:")
print(y_test.iloc[0])


# 15. SAVE MODEL


import joblib

joblib.dump(model, "insurance_linear_regression.pkl")
joblib.dump(scaler, "insurance_scaler.pkl")

print("\nModel Saved Successfully!")


DATASET SHAPE
(1338, 7)


FIRST 5 ROWS
   age     sex     bmi  children smoker     region      charges
0   19  female  27.900         0    yes  southwest  16884.92400
1   18    male  33.770         1     no  southeast   1725.55230
2   28    male  33.000         3     no  southeast   4449.46200
3   33    male  22.705         0     no  northwest  21984.47061
4   32    male  28.880         0     no  northwest   3866.85520


DATA TYPES
age           int64
sex          object
bmi         float64
children      int64
smoker       object
region       object
charges     float64
dtype: object


MISSING VALUES
age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64


DUPLICATES
1


STATISTICAL SUMMARY
               age          bmi     children       charges
count  1338.000000  1338.000000  1338.000000   1338.000000
mean     39.207025    30.663397     1.094918  13270.422265
std      14.049960     6.098187     1.205493  12110.011237
min      